In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

import pandas_datareader.data as web

In [ ]:
# Подготовка датасета для обучения в формате inputs, outputs
def make_datasets(input_data, n_inputs=2, n_outputs=1, gap=0):
	L = len(input_data)
	y = np.full((L-n_inputs-n_outputs-gap, n_outputs), 0.0)
	X = np.full((L-n_inputs-n_outputs-gap, n_inputs), 0.0)

	for i in range(n_inputs):
		X[:,i] = input_data[i:L-n_inputs-n_outputs-gap+i]

	for i in range(n_outputs):
		y[:,i] = input_data[n_inputs+gap+i:L-n_outputs+i]

	return X, y


In [ ]:
rate = web.DataReader(name='WGS10YR', data_source='fred', start='2000-01-01').diff().dropna()
rate.plot()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Переведем ряда массив Numpy
series = rate.iloc[:,0].to_numpy()

In [ ]:
# задаём ширину окна и горизонт прогнозирования
n_lags, fh= 20, 10

X, y = make_datasets(series, n_inputs=n_lags, n_outputs=fh)

In [ ]:
X_tensor = torch.Tensor(X).to(device)
y_tensor = torch.Tensor(y).to(device)

In [ ]:
train_dataset = TensorDataset(X_tensor, y_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
hidden_state = 30

model = nn.Sequential(
    nn.Linear(n_lags, hidden_state),
    nn.ReLU(),
    # nn.Sigmoid(),
    # nn.Tanh(),
    nn.Linear(hidden_state, fh)
).to(device)


In [ ]:
# Обучение
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())
epochs = 200


for epoch in range(epochs):
	total_loss = 0.0
	model.train()

	for batch_X, batch_y in train_loader:
		batch_X, batch_y = batch_X.to(device), batch_y.to(device)
		predictions = model(batch_X)
		loss = criterion(predictions, batch_y)

		loss.backward()
		optimizer.step()
		optimizer.zero_grad()

		total_loss += loss.item()

	print(f"Эпоха {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:
inputs = torch.Tensor(np.reshape(series[-n_lags:], (1, n_lags))).to(device)

model.eval()
with torch.no_grad():  # Отключаем вычисление градиентов
	outputs = model(inputs)

In [ ]:
y_pred = outputs.numpy()[0,]
y_pred

In [ ]:
last_obs = 50
plt.plot(np.arange(len(series)-last_obs, len(series)), series[-last_obs:])
plt.plot(np.arange(len(series), len(series)+fh), y_pred)
plt.show()